# Path-matching throughput benchmark

This notebook measures a repeated template-bank workflow: prepare many query
trees and templates once, then compute the complete query-template score matrix.

The notebook separates reusable preparation, first-use cost, warm score-matrix
time, objective-score accuracy, and process resources. Approximate methods may
return suboptimal matrices, but exact methods must agree entry by entry before
they are used as an oracle.

Three versioned profiles are available: `smoke`, `standard`, and `large`. The
checked-in default is now `large`; use `smoke` first on a new machine. See
`BENCHMARK_RUN_AND_FIGURE_GUIDE.pdf` in this folder for the recommended workflow.


Both `beam_partial_score` (the original generic beam) and `fast_beam_partial`
(the encoded beam) are included when `ALGORITHMS = "all"`. Nothing is replaced.
The comparison table checks paired scores as well as speed. Equality uses the
matched-budget comparison; finite-budget overlap uses different proposal rules.
See `FAST_BEAM_BENCHMARKS.md` for rerunning, figures, and preserving data in Git.


## Imports and repository setup

Run the notebook from anywhere inside the repository checkout. The bootstrap
below locates `pyproject.toml` and places `src/` on `sys.path`; an editable
installation is still recommended.

In [ ]:
from __future__ import annotations

from collections.abc import Mapping
from copy import deepcopy
from pathlib import Path
from time import perf_counter
import json
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_repository_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find the repository root. Run this notebook from inside the "
        "tree_matching checkout."
    )


def normalize_selection(value, *, available: list[str], setting_name: str) -> list[str]:
    if value == "all":
        selected = list(available)
    elif isinstance(value, str):
        selected = [value]
    else:
        selected = [str(item) for item in value]

    if not selected:
        raise ValueError(f"{setting_name} must select at least one item")
    if len(set(selected)) != len(selected):
        raise ValueError(f"{setting_name} contains duplicate entries: {selected}")

    unknown = sorted(set(selected).difference(available))
    if unknown:
        raise ValueError(
            f"Unknown {setting_name.lower()}: {unknown}. Available values are {available}."
        )

    selected_set = set(selected)
    return [name for name in available if name in selected_set]


def normalize_run_label(value: str | None) -> str | None:
    if value is None or not str(value).strip():
        return None
    label = str(value).strip()
    if re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", label) is None:
        raise ValueError(
            "RUN_LABEL must start with a letter or digit and contain only letters, "
            "digits, '.', '_', or '-'."
        )
    return label


ROOT = find_repository_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from benchmarking import (  # noqa: E402
    load_config,
    resolve_algorithm_specs,
    run_throughput_config,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
print(f"Repository root: {ROOT}")


## User settings

The controls below are deliberately compact. `SCALE` chooses a versioned preset;
`REGIMES` and `ALGORITHMS` select subsets without changing their preset order;
`SIZE_OVERRIDES` changes only tree size/depth fields. The large preset already
uses larger query/template banks, so global count overrides are normally left as
`None`.


In [ ]:
SCALE = "large"       # "smoke", "standard", or "large"
RUN_LABEL = "fast_beam_v2"  # change for each substantive run
REGIMES = "all"         # "all", one regime name, or a list of regime names
ALGORITHMS = "all"      # "all", one algorithm name, or a list of algorithm names
SIZE_OVERRIDES = {      # optional per-regime n/depth changes
    # "sparse_overlap_tree_template_bank": {"n_g": 8000, "n_h": 4000},
}
N_QUERIES = None        # None keeps each preset's regime-specific value
N_TEMPLATES = None      # None keeps each preset's regime-specific value
REPEATS = None          # None keeps the preset's timing repetitions
SEED = 20260928


## Resolve the preset

- `smoke` checks the APIs and output pipeline quickly;
- `standard` reproduces the first throughput matrix;
- `large` uses larger trees, larger query/template banks, and five timing repeats.

Algorithm parameters remain in named configurations. `SIZE_OVERRIDES` is limited
to `n_g`, `n_h`, `depth_g`, and `depth_h`; use the checked-in large preset rather
than manually tuning many workload parameters in the notebook.


In [ ]:
PRESET_PATHS = {
    "smoke": ROOT / "experiments" / "path_match_throughput_quick.json",
    "standard": ROOT / "experiments" / "path_match_throughput_full.json",
    "large": ROOT / "experiments" / "path_match_throughput_large.json",
}

scale_key = str(SCALE).strip().lower()
if scale_key not in PRESET_PATHS:
    raise ValueError(f"SCALE must be one of {sorted(PRESET_PATHS)}, not {SCALE!r}")

run_label = normalize_run_label(RUN_LABEL)
config = deepcopy(load_config(PRESET_PATHS[scale_key]))

all_regimes = list(config["regimes"])
available_regimes = [str(item["name"]) for item in all_regimes]
selected_names = normalize_selection(
    REGIMES,
    available=available_regimes,
    setting_name="REGIMES",
)

available_algorithms = [str(name) for name in config["algorithms"]]
selected_algorithms = normalize_selection(
    ALGORITHMS,
    available=available_algorithms,
    setting_name="ALGORITHMS",
)
config["algorithms"] = selected_algorithms

if SIZE_OVERRIDES is None:
    size_overrides = {}
elif isinstance(SIZE_OVERRIDES, Mapping):
    size_overrides = dict(SIZE_OVERRIDES)
else:
    raise TypeError("SIZE_OVERRIDES must be a mapping from regime name to overrides")

unknown_override_regimes = sorted(set(size_overrides).difference(available_regimes))
if unknown_override_regimes:
    raise ValueError(
        "SIZE_OVERRIDES contains unknown regimes: "
        f"{unknown_override_regimes}. Available regimes are {available_regimes}."
    )

allowed_size_fields = {"n_g", "n_h", "depth_g", "depth_h"}
normalized_overrides: dict[str, dict[str, int]] = {}
for regime_name, override in size_overrides.items():
    if not isinstance(override, Mapping):
        raise TypeError(f"SIZE_OVERRIDES[{regime_name!r}] must be a mapping")
    unknown_fields = sorted(set(override).difference(allowed_size_fields))
    if unknown_fields:
        raise ValueError(
            f"SIZE_OVERRIDES[{regime_name!r}] has unsupported fields {unknown_fields}; "
            f"allowed fields are {sorted(allowed_size_fields)}."
        )
    normalized = {}
    for field, value in override.items():
        integer_value = int(value)
        if integer_value < 1:
            raise ValueError(
                f"SIZE_OVERRIDES[{regime_name!r}][{field!r}] must be positive"
            )
        normalized[field] = integer_value
    normalized_overrides[str(regime_name)] = normalized

selected = set(selected_names)
resolved_regimes = []
for original_index, regime in enumerate(all_regimes):
    if regime["name"] not in selected:
        continue
    item = deepcopy(regime)
    # Stable under regime filtering: the offset is based on preset order.
    item["seed"] = int(SEED) + 1009 * original_index
    item.update(normalized_overrides.get(str(item["name"]), {}))
    if N_QUERIES is not None:
        if int(N_QUERIES) < 1:
            raise ValueError("N_QUERIES must be positive or None")
        item["n_queries"] = int(N_QUERIES)
    if N_TEMPLATES is not None:
        if int(N_TEMPLATES) < 1:
            raise ValueError("N_TEMPLATES must be positive or None")
        item["n_templates"] = int(N_TEMPLATES)
    resolved_regimes.append(item)

config["regimes"] = resolved_regimes
config["algorithm_order_seed"] = int(SEED)
base_suite_name = f"path_match_throughput_{scale_key}"
config["suite_name"] = (
    base_suite_name if run_label is None else f"{base_suite_name}_{run_label}"
)
if REPEATS is not None:
    if int(REPEATS) < 1:
        raise ValueError("REPEATS must be positive or None")
    config["repeats"] = int(REPEATS)

output_dir = ROOT / "benchmarking" / "results" / config["suite_name"]
if output_dir.exists() and any(output_dir.iterdir()):
    raise FileExistsError(
        f"Output directory is not empty: {output_dir}. Choose a new RUN_LABEL; "
        "existing results will not be overwritten."
    )


regime_columns = [
    "name", "n_queries", "n_templates", "n_g", "n_h", "depth_g", "depth_h",
    "comparison", "shape_g", "shape_h", "alphabet_size",
    "symbol_distribution", "symbols_per_node", "score_mode", "weight_mode",
    "expected_algorithm",
]
regime_table = pd.DataFrame(config["regimes"]).reindex(columns=regime_columns)

algorithm_specs = resolve_algorithm_specs(config["algorithms"])
algorithm_table = pd.DataFrame([
    {
        "algorithm": spec.name,
        "exact": spec.exact,
        "family": spec.family,
        "compatible_scores": ", ".join(spec.compatible_score_modes),
        "description": spec.description,
    }
    for spec in algorithm_specs
])

print(f"Preset: {PRESET_PATHS[scale_key].relative_to(ROOT)}")
print(f"Run label: {run_label or '(none)'}")
print(f"Output directory: {output_dir.relative_to(ROOT)}")
print(
    f"Timing repeats: {config['repeats']}; warm-up pairs: {config['warmup_pairs']}; "
    f"prepared/direct validation pairs: {config['validation_pairs']}"
)
print(f"Accuracy floor for the headline throughput comparison: {100 * config['quality_floor']:.1f}%")
if not any(spec.exact for spec in algorithm_specs):
    print("WARNING: no exact algorithm is selected; the run can measure scalability but not percent-of-optimum accuracy.")
print()
print("Selected throughput regimes")
display(regime_table)
print("Selected algorithm configurations")
display(algorithm_table)


## Run the throughput benchmark

For specialized and sparse methods, each tree is prepared once and that
representation is reused for the entire score matrix.  Timing repetitions
exclude corpus generation and reusable setup.  The `standard` preset runs each
algorithm/regime job in a fresh single-threaded subprocess, enforces a timeout,
and records lifetime peak resident memory; `smoke` remains an in-process wiring
check.  The backend saves row-level results, metadata, and the score matrices.


In [ ]:
started = perf_counter()
rows, summary, metadata = run_throughput_config(config, output_dir=output_dir)
elapsed = perf_counter() - started

print(f"Completed {len(rows):,} algorithm-regime runs in {elapsed:.2f} seconds.")
print(f"Rows:      {output_dir / 'throughput_rows.csv'}")
print(f"Summary:   {output_dir / 'throughput_summary.csv'}")
print(f"Matrices:  {output_dir / 'throughput_score_matrices.npz'}")

## Correctness, API, and oracle gates

The notebook stops if an implementation errors, a returned path fails validation,
a prepared API disagrees with its direct counterpart, exact score matrices
disagree beyond tolerance, or any method reports a score above the agreed exact
oracle. Intentional incompatibility/work-limit skips remain visible.

In [ ]:
status_table = (
    rows.groupby(["regime", "algorithm", "status"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
for column in ("ok", "skipped", "timeout", "error"):
    if column not in status_table:
        status_table[column] = 0
status_table = status_table[[
    "regime", "algorithm", "ok", "skipped", "timeout", "error"
]]
display(status_table)

error_rows = rows[rows["status"] == "error"]
if not error_rows.empty:
    display(error_rows[["regime", "algorithm", "error_type", "error_message"]])
    raise RuntimeError("At least one throughput algorithm raised an error")

timeout_rows = rows[rows["status"] == "timeout"]
if not timeout_rows.empty:
    display(timeout_rows[[
        "regime", "algorithm", "timeout_seconds", "fresh_process_total_seconds",
    ]])
    print("Timed-out methods remain as incomplete throughput results.")

prepared_failures = rows[
    (rows["status"] == "ok")
    & (rows["prepared_validation_applicable"] == True)  # noqa: E712
    & (rows["prepared_validation_passed"] != True)      # noqa: E712
]
if not prepared_failures.empty:
    display(prepared_failures[[
        "regime", "algorithm", "prepared_validation_max_score_difference",
        "prepared_validation_path_mismatches",
    ]])
    raise RuntimeError("A prepared matcher disagreed with its direct API")

invalid_paths = rows[
    (rows["status"] == "ok")
    & (
        (rows["validation_paths_valid"] != True)  # noqa: E712
        | (rows["validation_reported_scores_valid"] != True)  # noqa: E712
    )
]
if not invalid_paths.empty:
    display(invalid_paths[["regime", "algorithm"]])
    raise RuntimeError("A validation path or reported path score was invalid")

oracle_table = rows.drop_duplicates("regime")[[
    "regime", "oracle_status", "oracle_valid", "exact_method_count",
    "exact_matrix_max_abs_difference",
]]
display(oracle_table)

exact_disagreements = oracle_table[oracle_table["oracle_status"] == "exact_disagreement"]
if not exact_disagreements.empty:
    raise RuntimeError("Exact score matrices disagree; throughput accuracy is not interpretable")

score_violations = rows[
    (rows["status"] == "ok")
    & (rows["score_consistent_with_oracle"] == False)  # noqa: E712
]
if not score_violations.empty:
    display(score_violations[["regime", "algorithm", "matrix_score_sum"]])
    raise RuntimeError("A method exceeded the agreed exact score matrix")


## Throughput and accuracy summary

`pairs_per_second` excludes one-time reusable setup. `setup_plus_one_matrix_seconds`
shows the end-to-end cost of preparing the corpus for one algorithm and computing
one score matrix. The headline method must meet the accuracy floor on every
query-template pair. The harmonic score is secondary.

In [ ]:
regime_summary = (
    summary.sort_values(["regime_order", "algorithm_config_index"])
    .drop_duplicates("regime")[[
        "regime", "expected_algorithm", "oracle_status",
        "observed_fastest_quality_eligible", "observed_best_harmonic_score",
        "quality_floor",
    ]]
    .reset_index(drop=True)
)
regime_summary["quality_floor_percent"] = 100.0 * regime_summary.pop("quality_floor")
display(regime_summary)

compact_columns = [
    "regime", "regime_order", "algorithm", "algorithm_config_index", "status",
    "preparation_kind", "one_time_preparation_seconds",
    "score_matrix_seconds_median", "score_matrix_seconds_q25",
    "score_matrix_seconds_q75", "score_matrix_relative_iqr",
    "setup_plus_one_matrix_seconds", "cold_score_matrix_seconds",
    "cold_setup_plus_one_matrix_seconds", "fresh_process_total_seconds",
    "peak_rss_mib", "pairs_per_second", "queries_per_second",
    "matrix_total_accuracy", "mean_pair_accuracy", "min_pair_accuracy",
    "exact_pair_fraction", "throughput_timing_score",
    "throughput_accuracy_harmonic_score", "quality_eligible",
    "throughput_frontier",
]
compact = summary[[c for c in compact_columns if c in summary.columns]].copy()
for source, target in (
    ("matrix_total_accuracy", "matrix_total_accuracy_percent"),
    ("mean_pair_accuracy", "mean_pair_accuracy_percent"),
    ("min_pair_accuracy", "min_pair_accuracy_percent"),
    ("exact_pair_fraction", "exact_pair_percent"),
):
    compact[target] = 100.0 * pd.to_numeric(compact[source], errors="coerce")
compact["score_matrix_iqr_seconds"] = (
    pd.to_numeric(compact["score_matrix_seconds_q75"], errors="coerce")
    - pd.to_numeric(compact["score_matrix_seconds_q25"], errors="coerce")
)
compact = compact.drop(columns=[
    "matrix_total_accuracy", "mean_pair_accuracy", "min_pair_accuracy",
    "exact_pair_fraction",
])
display(compact.sort_values(["regime_order", "score_matrix_seconds_median"]))


## Warm throughput versus score-matrix accuracy

Each point is one algorithm preset. Regimes without an exact score matrix do not
receive an accuracy plot. Frontier points are not dominated simultaneously in
warm matrix time and aggregate score-matrix accuracy.

In [ ]:
for regime_name in [item["name"] for item in config["regimes"]]:
    plot_data = compact[
        (compact["regime"] == regime_name)
        & compact["pairs_per_second"].notna()
        & compact["matrix_total_accuracy_percent"].notna()
        & (compact["pairs_per_second"] > 0.0)
    ].copy()
    if plot_data.empty:
        print(f"{regime_name}: no exact-oracle accuracy plot")
        continue

    fig = plt.figure(figsize=(8, 5))
    ax = fig.add_subplot(111)
    ax.scatter(plot_data["pairs_per_second"], plot_data["matrix_total_accuracy_percent"])
    for _, point in plot_data.iterrows():
        marker = " *" if bool(point["throughput_frontier"]) else ""
        ax.annotate(
            f"{point['algorithm']}{marker}",
            (point["pairs_per_second"], point["matrix_total_accuracy_percent"]),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
        )
    ax.axhline(100.0 * float(config["quality_floor"]), linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_xlabel("Warm query-template pairs per second (log scale)")
    ax.set_ylabel("Aggregate score-matrix accuracy (%)")
    ax.set_title(regime_name)
    ax.grid(True, alpha=0.25)
    plt.show()

## One-time preparation, warm search, and process resources

Reusable preprocessing can change the relative ordering when one template bank
is queried repeatedly.  This table keeps setup components separate, and the
standard preset additionally reports fresh-process total time and peak resident
memory rather than folding them into warm throughput.


In [ ]:
setup_columns = [
    "regime", "algorithm", "matcher_construction_seconds", "encoder_fit_seconds",
    "query_preparation_seconds", "template_preparation_seconds",
    "one_time_preparation_seconds", "score_matrix_seconds_median",
    "score_matrix_seconds_q25", "score_matrix_seconds_q75",
    "setup_plus_one_matrix_seconds", "cold_one_time_preparation_seconds",
    "cold_score_matrix_seconds", "cold_setup_plus_one_matrix_seconds",
    "fresh_process_total_seconds",
    "peak_rss_mib",
]
setup_columns = [column for column in setup_columns if column in summary.columns]
setup_table = summary[summary["status"] == "ok"][setup_columns].copy()
display(setup_table.sort_values(["regime", "setup_plus_one_matrix_seconds"]))


## Sampled algorithmic work counters

Diagnostics are collected on a small deterministic sample of pairs outside the
clean timing loop. Blank entries mean that a counter is not meaningful for that
algorithm.

In [ ]:
diagnostic_columns = [
    column for column in summary.columns
    if column.startswith("sample_median_diag_")
    and summary[column].notna().any()
]
work_table = summary[[
    "regime", "algorithm", "diagnostic_pairs_checked", *diagnostic_columns
]].copy()
display(work_table)

## Save notebook-facing tables

The backend has already saved the complete result rows, metadata, and compressed
score matrices. This cell adds the compact tables displayed above and a copy of
the exact notebook configuration.

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
compact.to_csv(output_dir / "notebook_throughput_table.csv", index=False)
regime_summary.to_csv(output_dir / "notebook_regime_summary.csv", index=False)
setup_table.to_csv(output_dir / "notebook_setup_table.csv", index=False)
work_table.to_csv(output_dir / "notebook_work_summary.csv", index=False)
(output_dir / "notebook_config.json").write_text(
    json.dumps(config, indent=2, sort_keys=True, default=str),
    encoding="utf-8",
)

print("Saved:")
for name in (
    "throughput_rows.csv",
    "throughput_summary.csv",
    "throughput_metadata.json",
    "throughput_score_matrices.npz",
    "notebook_throughput_table.csv",
    "notebook_regime_summary.csv",
    "notebook_setup_table.csv",
    "notebook_work_summary.csv",
    "notebook_config.json",
):
    print(f"  {output_dir / name}")

## Reading the result

Warm throughput answers the operational question after the template bank and
queries have been prepared. `setup_plus_one_matrix_seconds` answers whether that
preparation is worthwhile for a single score matrix.  Fresh-process total time
and peak resident memory are separate deployment-oriented measurements.  For a
recurring template bank, the warm result becomes increasingly relevant; for
one-off comparisons, the pairwise ranking notebook is the better summary.


## Produce talk-ready figures

After the pairwise and throughput notebooks (and optionally the sweep runner)
finish, run the shared plotting script described in
`benchmarking/BENCHMARK_RUN_AND_FIGURE_GUIDE.pdf`. It writes uncluttered
speed-accuracy panels, exact generic-versus-encoded comparisons, scaling curves,
and beam-parameter tradeoff figures as both PNG and PDF.


## Generic versus encoded: exact DP and partial beam

These tables compare the original and accelerated implementations on the same
instances. Speedups are generic time divided by encoded time. The paired score
check is not a claim of optimality. In overlap mode the two finite-budget beam
proposal rules differ, so score differences must be interpreted together with time.


In [ ]:
implementation_table = pd.read_csv(output_dir / "implementation_comparison_summary.csv")
display(implementation_table)
print("Paired scores and speedups saved in implementation_comparison_rows.csv")
print("Data are local until explicitly archived and committed; see FAST_BEAM_BENCHMARKS.md.")
